# R-PROP2: Proposition 4.2 Expansion (n=250) -- Colab Notebook

## Cell 0: Setup

In [ ]:
# Clone repo + mount Drive
!git clone https://github.com/melodiz/rbpo.git /content/rbpo 2>/dev/null || \
    (cd /content/rbpo && git pull origin main)
%cd /content/rbpo

from google.colab import drive
drive.mount('/content/drive')

## Cell 1: Install dependencies

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")

!pip install -q editdistance sentencepiece lhotse kaldialign

# Match k2 to Colab's native PyTorch + CUDA
# Check torch version above and pick the matching wheel.
# See notebooks/troubleshooting.md Sec.3 if this breaks torch.
!pip install -q k2==1.24.4.dev20260306+cuda12.8.torch2.10.0 \
    -f https://k2-fsa.github.io/k2/cuda.html 2>/dev/null || \
    pip install -q k2

# Verify k2 loaded (k2 has no __version__ attribute)
import k2
print(f"k2 loaded: {hasattr(k2, 'ctc_topo')}")

## Cell 2: Download model checkpoint

In [ ]:
import os
from pathlib import Path

MODEL_DIR = Path("/content/icefall-asr-librispeech-zipformer-small-cr-ctc")
CHECKPOINT_PATH = MODEL_DIR / "exp" / "pretrained.pt"
BPE_MODEL_PATH  = MODEL_DIR / "data" / "lang_bpe_500" / "bpe.model"

if not CHECKPOINT_PATH.exists():
    HF_BASE = "https://huggingface.co/Zengwei/icefall-asr-librispeech-zipformer-small-cr-ctc-20241018/resolve/main"
    os.makedirs(MODEL_DIR / "exp", exist_ok=True)
    os.makedirs(MODEL_DIR / "data" / "lang_bpe_500", exist_ok=True)
    print("Downloading pretrained.pt (~89 MB)...")
    !wget -q -L --show-progress -O {CHECKPOINT_PATH} "{HF_BASE}/exp/pretrained.pt"
    print("Downloading bpe.model...")
    !wget -q -L --show-progress -O {BPE_MODEL_PATH} "{HF_BASE}/data/lang_bpe_500/bpe.model"
else:
    print("Model checkpoint already present")

ckpt_size = CHECKPOINT_PATH.stat().st_size
assert ckpt_size > 1_000_000, f"Checkpoint too small ({ckpt_size} bytes)  --  download may have failed"
print(f"Checkpoint: {ckpt_size / 1e6:.1f} MB")

## Cell 3: Clone icefall + prepare data

In [ ]:
import glob

ICEFALL_DIR = Path("/content/icefall")
DATA_DIR = Path("/content/librispeech_data")
CUTS_PATH = DATA_DIR / "cuts" / "librispeech_cuts_dev-other.jsonl.gz"

# Clone icefall (needed for model architecture code)
if not ICEFALL_DIR.exists():
    print("Cloning icefall...")
    !git clone --depth 1 https://github.com/k2-fsa/icefall.git {ICEFALL_DIR}
    !cd {ICEFALL_DIR} && pip install -q -e .
else:
    print("icefall already present")

# --- Resolve cuts ---
DRIVE_CUTS = Path("/content/drive/MyDrive/rbpo_results/librispeech_cuts/librispeech_cuts_dev-other.jsonl.gz")
DRIVE_CUTS_ALT = Path("/content/drive/MyDrive/rbpo_results/cuts/librispeech_cuts_dev-other.jsonl.gz")

if not CUTS_PATH.exists():
    os.makedirs(CUTS_PATH.parent, exist_ok=True)
    import shutil
    for src in [DRIVE_CUTS, DRIVE_CUTS_ALT]:
        if src.exists():
            shutil.copy2(str(src), str(CUTS_PATH))
            print(f"Cuts copied from Drive: {src}")
            break

# --- Check if cuts have pre-computed features ---
has_features = False
if CUTS_PATH.exists():
    from lhotse import load_manifest_lazy
    first_cut = next(iter(load_manifest_lazy(str(CUTS_PATH))))
    has_features = first_cut.load_features() is not None

# --- Ensure audio is present when features aren't pre-computed ---
# (troubleshooting.md Sec.5: cuts cached from Drive reference local audio paths
#  that don't exist on this instance)
LS_AUDIO_DIR = DATA_DIR / "LibriSpeech"
needs_audio = CUTS_PATH.exists() and not has_features
needs_full_prep = not CUTS_PATH.exists()

if needs_audio or needs_full_prep:
    if not glob.glob(str(LS_AUDIO_DIR / "dev-other" / "**" / "*.flac"), recursive=True):
        print("Downloading LibriSpeech dev-other audio (~314 MB)...")
        os.makedirs(DATA_DIR, exist_ok=True)
        !wget -q --show-progress -O {DATA_DIR}/dev-other.tar.gz \
            "https://www.openslr.org/resources/12/dev-other.tar.gz"
        !cd {DATA_DIR} && tar xzf dev-other.tar.gz && rm -f dev-other.tar.gz
    else:
        print("Audio .flac files already present")

if needs_full_prep:
    # Build cuts from scratch (audio + manifests + CutSet)
    from lhotse.recipes.librispeech import prepare_librispeech
    from lhotse import CutSet
    LS_MANIFESTS_DIR = Path("/content/librispeech_manifests")
    os.makedirs(LS_MANIFESTS_DIR, exist_ok=True)

    print("Preparing lhotse manifests...")
    manifests = prepare_librispeech(
        corpus_dir=LS_AUDIO_DIR,
        dataset_parts=["dev-other"],
        output_dir=LS_MANIFESTS_DIR,
        num_jobs=2,
    )
    cuts = CutSet.from_manifests(
        recordings=manifests["dev-other"]["recordings"],
        supervisions=manifests["dev-other"]["supervisions"],
    )
    cuts = cuts.trim_to_supervisions().to_eager()
    os.makedirs(CUTS_PATH.parent, exist_ok=True)
    cuts.to_file(str(CUTS_PATH))
    print(f"CutSet saved: {len(cuts)} utterances")

# --- Final verification ---
from lhotse import load_manifest_lazy
test_cuts = load_manifest_lazy(str(CUTS_PATH))
first_cut = next(iter(test_cuts))
print(f"\nVerification: cut '{first_cut.id}', duration={first_cut.duration:.1f}s")

feats = first_cut.load_features()
if feats is not None:
    print(f"Features: pre-computed fbank, shape={feats.shape}")
else:
    audio = first_cut.load_audio()
    print(f"Features: none (script extracts fbank on-the-fly)")
    print(f"Audio: shape={audio.shape}, sr={first_cut.recording.sampling_rate}")

## Cell 4: Run R-PROP2 experiment

In [ ]:
!python /content/rbpo/scripts/prop42_expansion.py \
    --model-dir /content/icefall-asr-librispeech-zipformer-small-cr-ctc \
    --icefall-dir /content/icefall \
    --data-dir /content/librispeech_data \
    --output-dir /content/drive/MyDrive/rbpo_results/R_prop2_expansion \
    --gamma-csv /content/drive/MyDrive/rbpo_results/gap_covering/stage2/gamma_stats.csv \
    --n-utterances 250 \
    --G 8 \
    --device cuda:0

## Cell 5: Inspect results

In [ ]:
import json

with open("/content/drive/MyDrive/rbpo_results/R_prop2_expansion/prop42_results.json") as f:
    results = json.load(f)

print("=== R-PROP2 Summary ===")
print(f"Valid utterances: {results['n_utts_valid']}")
print(f"Ordering violations: {results['n_violations']}")
print()
print(f"Viterbi/CTC: {results['mean_ratio_viterbi']:.4f} +/- {results['sd_ratio_viterbi']:.4f}")
print(f"  95% CI: [{results['ci_lower_viterbi']:.4f}, {results['ci_upper_viterbi']:.4f}]")
print(f"Sampled/CTC: {results['mean_ratio_sampled']:.4f} +/- {results['sd_ratio_sampled']:.4f}")
print(f"  95% CI: [{results['ci_lower_sampled']:.4f}, {results['ci_upper_sampled']:.4f}]")

if "entropy_correlation" in results:
    ec = results["entropy_correlation"]
    print(f"\nEntropy correlation (n={ec['n_overlap']}):")
    print(f"  rho(entropy, viterbi/ctc) = {ec['spearman_entropy_vs_viterbi_ratio']:.3f} (p={ec['p_value_viterbi']:.4f})")
    print(f"  rho(entropy, sampled/ctc) = {ec['spearman_entropy_vs_sampled_ratio']:.3f} (p={ec['p_value_sampled']:.4f})")

## Cell 6: List bring-back files on Drive

In [ ]:
# All outputs are on Drive  --  download these locally after the session
print("Bring-back files (download from Drive):")
!ls -la /content/drive/MyDrive/rbpo_results/R_prop2_expansion/